#About Dataset

link https://www.kaggle.com/code/emekaydn/wine-quality

Context


The two datasets are related to red and white variants of the Portuguese "Vinho Verde" wine. For more details, consult the reference [Cortez et al., 2009]. Due to privacy and logistic issues, only physicochemical (inputs) and sensory (the output) variables are available (e.g. there is no data about grape types, wine brand, wine selling price, etc.).

These datasets can be viewed as classification or regression tasks. The classes are ordered and not balanced (e.g. there are much more normal wines than excellent or poor ones).

##Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import layers, models

##Load the data

In [2]:
df = pd.read_csv("winequality-red.csv")

In [3]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


##Explore the data

you can see target col is really imbalaanced so we need to select good values to get good results

In [4]:
df["quality"].value_counts()

,count
quality,
5,681
6,638
7,199
4,53
8,18
3,10


In [5]:
df.shape

(1599, 12)

In [6]:
df.isnull().sum().sum()

np.int64(0)

In [7]:
df.duplicated().sum()

np.int64(240)

##Preprocessing

In [8]:
df["target"] = df['quality'].apply(lambda x : 1 if x >= 7 else 0)

In [9]:
df["target"].value_counts()

,count
target,
0,1382
1,217


In [10]:
df.drop_duplicates(inplace=True)
display(df.duplicated().sum())

np.int64(0)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1359 entries, 0 to 1598
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1359 non-null   float64
 1   volatile acidity      1359 non-null   float64
 2   citric acid           1359 non-null   float64
 3   residual sugar        1359 non-null   float64
 4   chlorides             1359 non-null   float64
 5   free sulfur dioxide   1359 non-null   float64
 6   total sulfur dioxide  1359 non-null   float64
 7   density               1359 non-null   float64
 8   pH                    1359 non-null   float64
 9   sulphates             1359 non-null   float64
 10  alcohol               1359 non-null   float64
 11  quality               1359 non-null   int64  
 12  target                1359 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 148.6 KB


In [12]:
df = df.drop(columns=["quality"])

In [13]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,target
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,0
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,0


In [14]:
df["target"].value_counts()

,count
target,
0,1175
1,184


##Training and Building the Model

In [15]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']


In [16]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=32, stratify=y_temp
)

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

Handle Imbalance

In [18]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = {i: w for i, w in enumerate(cw)}
class_weight

{0: np.float64(0.5781914893617022), 1: np.float64(3.697278911564626)}

In [26]:
def build_model(input_dims):

  inputs = layers.Input(shape=(input_dims,))
  x = layers.Dense(128,activation="relu")(inputs)
  x = layers.Dropout(0.3)(x)
  x = layers.Dense(62,activation="relu")(x)
  x = layers.Dropout(0.3)(x)
  outputs = layers.Dense(1, activation='sigmoid')(x)

  model = tf.keras.Model(inputs,outputs)

  model.compile(
                optimizer='adam', # Added an optimizer, you can change this
                loss = 'binary_crossentropy',
                metrics=['accuracy',
                         tf.keras.metrics.AUC(name="auc"),
                          tf.keras.metrics.Precision(name="precision"),
                          tf.keras.metrics.Recall(name="recall")])
  return model

In [28]:
model = build_model(X_train.shape[1])

In [29]:
history = model.fit(
    X_train,y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs = 44,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping('val_loss', patience=3,
                                               restore_best_weights=True)]
)

Epoch 1/44
34/34 ━━━━━━━━━━━━━━━━━━━━ 8s 103ms/step - accuracy: 0.8363 - auc: 0.5764 - loss: 0.6357 - precision: 0.2238 - recall: 0.1959 - val_accuracy: 0.7794 - val_auc: 0.9160 - val_loss: 0.5206 - val_precision: 0.3696 - val_recall: 0.9444
Epoch 2/44
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7361 - auc: 0.8623 - loss: 0.5117 - precision: 0.3360 - recall: 0.8404 - val_accuracy: 0.8088 - val_auc: 0.9211 - val_loss: 0.4225 - val_precision: 0.4048 - val_recall: 0.9444
Epoch 3/44
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7826 - auc: 0.8831 - loss: 0.4234 - precision: 0.3417 - recall: 0.8127 - val_accuracy: 0.8309 - val_auc: 0.9334 - val_loss: 0.3973 - val_precision: 0.4390 - val_recall: 1.0000
Epoch 4/44
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7717 - auc: 0.8844 - loss: 0.4354 - precision: 0.3923 - recall: 0.9061 - val_accuracy: 0.8456 - val_auc: 0.9416 - val_loss: 0.3599 - val_precision: 0.4615 - val_recall: 1.0000
Epoch 5/44
34/34 ━━━━━━━━━━━━━━━━━━━━

In [30]:
from sklearn.metrics import classification_report, confusion_matrix

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred, digits=4))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Confusion Matrix:
 [[87 30]
 [ 6 13]]

Report:
               precision    recall  f1-score   support

           0     0.9355    0.7436    0.8286       117
           1     0.3023    0.6842    0.4194        19

    accuracy                         0.7353       136
   macro avg     0.6189    0.7139    0.6240       136
weighted avg     0.8470    0.7353    0.7714       136



##Tuning

In [31]:
from sklearn.metrics import precision_recall_curve, f1_score

# Get validation probabilities
val_probs = model.predict(X_val).ravel()

prec, rec, thr = precision_recall_curve(y_val, val_probs)

# Compute F1 for each threshold
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = f1s.argmax()
best_thr = thr[max(best_idx-1, 0)]  # careful: thr has length len(prec)-1

print("Best threshold:", best_thr)
print("Best F1 on val:", f1s[best_idx])

# Re-evaluate on test set with new threshold
test_probs = model.predict(X_test).ravel()
y_pred_new = (test_probs >= best_thr).astype(int)

print("\nNew Confusion Matrix:\n", confusion_matrix(y_test, y_pred_new))
print("\nNew Report:\n", classification_report(y_test, y_pred_new, digits=4))


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
Best threshold: 0.7086261
Best F1 on val: 0.7368421047645428
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

New Confusion Matrix:
 [[102  15]
 [  7  12]]

New Report:
               precision    recall  f1-score   support

           0     0.9358    0.8718    0.9027       117
           1     0.4444    0.6316    0.5217        19

    accuracy                         0.8382       136
   macro avg     0.6901    0.7517    0.7122       136
weighted avg     0.8671    0.8382    0.8494       136



In [32]:
!pip -q install imbalanced-learn


In [33]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# X = df.drop(columns=['quality']).values.astype('float32')
# y = (df['quality'] >= 7).astype(int).values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit on train only
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)


In [34]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

print("Before:", Counter(y_train))
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)
print("After :", Counter(y_train_ros))


Before: Counter({0: 940, 1: 147})
After : Counter({0: 940, 1: 940})


In [35]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# choose a safe k for minority size
minority_count = sum(y_train==1)
k = max(1, min(5, minority_count-1))  # SMOTE needs k_neighbors < minority_count

print("Before:", Counter(y_train))
sm = SMOTE(random_state=42, k_neighbors=k)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print("After :", Counter(y_train_sm))


Before: Counter({0: 940, 1: 147})
After : Counter({0: 940, 1: 940})


In [36]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_binary_mlp(input_dim):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inp, out)
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                           tf.keras.metrics.Precision(name='precision'),
                           tf.keras.metrics.Recall(name='recall')])
    return model

model = build_binary_mlp(X_train.shape[1])


In [37]:
history = model.fit(X_train_sm, y_train_sm, validation_data=(X_val, y_val),
                    epochs=30, batch_size=32,
                    callbacks=[tf.keras.callbacks.EarlyStopping('val_loss', patience=3, restore_best_weights=True)])

Epoch 1/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 72ms/step - accuracy: 0.6919 - auc: 0.7597 - loss: 0.5780 - precision: 0.6882 - recall: 0.7053 - val_accuracy: 0.8382 - val_auc: 0.8772 - val_loss: 0.3777 - val_precision: 0.4516 - val_recall: 0.7368
Epoch 2/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8182 - auc: 0.8846 - loss: 0.4242 - precision: 0.8036 - recall: 0.8431 - val_accuracy: 0.8456 - val_auc: 0.8695 - val_loss: 0.3647 - val_precision: 0.4688 - val_recall: 0.7895
Epoch 3/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8240 - auc: 0.9028 - loss: 0.3888 - precision: 0.8081 - recall: 0.8505 - val_accuracy: 0.8676 - val_auc: 0.8646 - val_loss: 0.3367 - val_precision: 0.5185 - val_recall: 0.7368
Epoch 4/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8437 - auc: 0.9126 - loss: 0.3654 - precision: 0.8355 - recall: 0.8700 - val_accuracy: 0.8824 - val_auc: 0.8637 - val_loss: 0.3548 - val_precision: 0.5556 - val_recall: 0.7895
Epoch 5/30
59/59 ━━━━━━━━━━━━━━━━━━━━ 0

In [38]:
from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix, roc_auc_score, average_precision_score
import numpy as np

val_probs = model.predict(X_val, verbose=0).ravel()
prec, rec, thr = precision_recall_curve(y_val, val_probs)
f1s = 2*prec*rec/(prec+rec+1e-9)
best_idx = f1s.argmax()
best_thr = thr[max(best_idx-1, 0)]
print("Best threshold on val:", best_thr, " | best F1:", f1s[best_idx])

test_probs = model.predict(X_test, verbose=0).ravel()
y_pred = (test_probs >= best_thr).astype(int)

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, test_probs))
print("PR-AUC :", average_precision_score(y_test, test_probs))


Best threshold on val: 0.75370514  | best F1: 0.7058823524480968

Confusion Matrix:
 [[102  16]
 [  8  10]]

Report:
               precision    recall  f1-score   support

           0     0.9273    0.8644    0.8947       118
           1     0.3846    0.5556    0.4545        18

    accuracy                         0.8235       136
   macro avg     0.6559    0.7100    0.6746       136
weighted avg     0.8555    0.8235    0.8365       136

ROC-AUC: 0.8709981167608287
PR-AUC : 0.5042861961428121
